## Árvore de Decisão

A [Árvore de Decisão](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) é um modelo baseado em regras, que realiza divisões sucessivas nos dados com o objetivo de separar as classes.

Cada divisão é feita com base em uma variável e um ponto de corte, formando uma estrutura semelhante a uma árvore.

Esse modelo é interpretável e capaz de capturar relações não lineares entre as variáveis.

### Hiperparâmetros utilizados

- **max_depth**: define a profundidade máxima da árvore.
  - Valores menores → modelo mais simples (menos overfitting)
  - Valor `None` → árvore cresce livremente

- **min_samples_split**: número mínimo de amostras necessárias para dividir um nó.
  - Valores maiores tornam o modelo mais conservador

- **min_samples_leaf**: número mínimo de amostras em uma folha.
  - Evita que a árvore crie divisões muito específicas

- **criterion**: métrica utilizada para avaliar a qualidade da divisão.
  - `gini`: índice de Gini (default)
  - `entropy`: ganho de informação

### Estratégia

Foi utilizado o GridSearchCV com validação cruzada (5 folds) para ajustar os hiperparâmetros e reduzir o risco de overfitting.

In [1]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    import pandas as pd
    import sys
    import os

    # add raiz do projeto
    sys.path.append(os.path.abspath(".."))

    from sklearn.model_selection import GridSearchCV
    from imblearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTE
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

    from preprocessing.main_preprocessing import load_and_preprocess
    from utils.experiment_logger import save_results


save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade

🔎 Scenario: submodalidade_agrupada

🔎 Scenario: submodalidade_engineered


""


## BASELINE

In [2]:
# scenarios = [
#     "sem_submodalidade",
#     "submodalidade_agrupada",
#     "submodalidade_engineered"
# ]

# smote_options = [False, True]

# results = []


# for scenario in scenarios:
#     for use_smote in smote_options:

#         print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

#         X_train, X_test, y_train, y_test = load_and_preprocess(
#             "../predictive_models/scrdata_202505.csv",
#             scenario=scenario,
#             use_smote=use_smote
#         )

#         steps = []

#         if use_smote:
#             steps.append(('smote', SMOTE(random_state=42)))
#         steps.append(('dt', DecisionTreeClassifier(random_state=42)))

#         model = Pipeline(steps)

#         model.fit(X_train, y_train)

#         y_pred = model.predict(X_test)
#         y_proba = model.predict_proba(X_test)[:, 1]

#         results.append({
#             "model": "DecisionTree",
#             "scenario": scenario,
#             "smote": use_smote,
#             "roc_auc": roc_auc_score(y_test, y_proba),
#             "f1": f1_score(y_test, y_pred),
#             "accuracy": accuracy_score(y_test, y_pred),
#             "n_features": X_train.shape[1],
#             "phase": "baseline",
#             "timestamp": pd.Timestamp.now()
#         })


# save_results(results, file_path="../results/experiments.csv")

# df_result = pd.DataFrame(results)

# display(df_result)


## GRIDSEARCH

In [3]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    X_train, X_test, y_train, y_test = load_and_preprocess(
        "../predictive_models/scrdata_202505.csv",
        scenario=scenario,
        use_smote=False
    )


    param_grid_dt = {
        "smote": [SMOTE(random_state=42), "passthrough"],
        "dt__max_depth": [10, 20, 30],
        "dt__min_samples_split": [2, 5, 10],
        "dt__min_samples_leaf": [1, 2, 4],
        "dt__criterion": ["gini", "entropy"]
    }

    grid_dt = GridSearchCV(
        estimator=Pipeline([
            ('smote', SMOTE(random_state=42)),
            ('dt', DecisionTreeClassifier(random_state=42))
        ]),
        param_grid=param_grid_dt,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )

    grid_dt.fit(X_train, y_train)

    print("Best parameters:", grid_dt.best_params_)
    print("Best ROC AUC (CV):", grid_dt.best_score_)


    #? BEST MODEL TEST EVALUATION

    best_dt = grid_dt.best_estimator_

    y_pred = best_dt.predict(X_test)
    y_proba = best_dt.predict_proba(X_test)[:, 1]


    #? TUNING (CV)



    cv_results = pd.DataFrame(grid_dt.cv_results_)
    cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

    for smote_val in [True, False]:
        sub_results = cv_results[cv_results['smote_used'] == smote_val]
        if not sub_results.empty:
            best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
            params = best_row['params']

            tuning_results.append({
                "model": "DecisionTree",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "tuning_cv",
                "roc_auc": best_row['mean_test_score'],
                "f1": None,
                "accuracy": None,
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

            # Re-fit the best model for this group to get test metrics
            best_model_group = grid_dt.estimator.set_params(**params)
            best_model_group.fit(X_train, y_train)

            y_pred = best_model_group.predict(X_test)
            y_proba = best_model_group.predict_proba(X_test)[:, 1]

            tuning_results.append({
                "model": "DecisionTree",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "test",
                "roc_auc": roc_auc_score(y_test, y_proba),
                "f1": f1_score(y_test, y_pred),
                "accuracy": accuracy_score(y_test, y_pred),
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

    save_results(tuning_results, file_path="../results/model_results.csv")

    display(pd.DataFrame(tuning_results))

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade
Best parameters: {'dt__criterion': 'entropy', 'dt__max_depth': 10, 'dt__min_samples_leaf': 4, 'dt__min_samples_split': 10, 'smote': 'passthrough'}
Best ROC AUC (CV): 0.9113779414410601


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,DecisionTree,sem_submodalidade,True,tuning_cv,0.908520,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:52.477156
1,DecisionTree,sem_submodalidade,True,test,0.907896,0.831701,0.817412,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:57.829911
2,DecisionTree,sem_submodalidade,False,tuning_cv,0.911378,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:57.830729
3,DecisionTree,sem_submodalidade,False,test,0.910683,0.854567,0.830973,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:59.405336



🔎 Scenario: submodalidade_agrupada
Best parameters: {'dt__criterion': 'entropy', 'dt__max_depth': 10, 'dt__min_samples_leaf': 4, 'dt__min_samples_split': 10, 'smote': 'passthrough'}
Best ROC AUC (CV): 0.9224050761930933


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,DecisionTree,sem_submodalidade,True,tuning_cv,0.908520,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:52.477156
1,DecisionTree,sem_submodalidade,True,test,0.907896,0.831701,0.817412,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:57.829911
2,DecisionTree,sem_submodalidade,False,tuning_cv,0.911378,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:57.830729
3,DecisionTree,sem_submodalidade,False,test,0.910683,0.854567,0.830973,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:59.405336
4,DecisionTree,submodalidade_agrupada,True,tuning_cv,0.919749,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:09:59.104656
5,DecisionTree,submodalidade_agrupada,True,test,0.918547,0.850267,0.832962,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:10:05.422844
6,DecisionTree,submodalidade_agrupada,False,tuning_cv,0.922405,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:10:05.423682
7,DecisionTree,submodalidade_agrupada,False,test,0.922099,0.870189,0.845172,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:10:07.120320



🔎 Scenario: submodalidade_engineered
Best parameters: {'dt__criterion': 'entropy', 'dt__max_depth': 10, 'dt__min_samples_leaf': 4, 'dt__min_samples_split': 10, 'smote': 'passthrough'}
Best ROC AUC (CV): 0.9113779414410601


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,DecisionTree,sem_submodalidade,True,tuning_cv,0.908520,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:52.477156
1,DecisionTree,sem_submodalidade,True,test,0.907896,0.831701,0.817412,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:57.829911
2,DecisionTree,sem_submodalidade,False,tuning_cv,0.911378,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:57.830729
3,DecisionTree,sem_submodalidade,False,test,0.910683,0.854567,0.830973,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:59.405336
4,DecisionTree,submodalidade_agrupada,True,tuning_cv,0.919749,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:09:59.104656
5,DecisionTree,submodalidade_agrupada,True,test,0.918547,0.850267,0.832962,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:10:05.422844
6,DecisionTree,submodalidade_agrupada,False,tuning_cv,0.922405,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:10:05.423682
7,DecisionTree,submodalidade_agrupada,False,test,0.922099,0.870189,0.845172,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:10:07.120320
8,DecisionTree,submodalidade_engineered,True,tuning_cv,0.908520,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:20:26.414230
9,DecisionTree,submodalidade_engineered,True,test,0.907896,0.831701,0.817412,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:20:31.466166


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,DecisionTree,sem_submodalidade,True,tuning_cv,0.908520,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:52.477156
1,DecisionTree,sem_submodalidade,True,test,0.907896,0.831701,0.817412,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:57.829911
2,DecisionTree,sem_submodalidade,False,tuning_cv,0.911378,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:57.830729
3,DecisionTree,sem_submodalidade,False,test,0.910683,0.854567,0.830973,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 22:56:59.405336
4,DecisionTree,submodalidade_agrupada,True,tuning_cv,0.919749,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:09:59.104656
5,DecisionTree,submodalidade_agrupada,True,test,0.918547,0.850267,0.832962,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:10:05.422844
6,DecisionTree,submodalidade_agrupada,False,tuning_cv,0.922405,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:10:05.423682
7,DecisionTree,submodalidade_agrupada,False,test,0.922099,0.870189,0.845172,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:10:07.120320
8,DecisionTree,submodalidade_engineered,True,tuning_cv,0.908520,NaN,NaN,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:20:26.414230
9,DecisionTree,submodalidade_engineered,True,test,0.907896,0.831701,0.817412,"{'dt__criterion': 'entropy', 'dt__max_depth': ...",2026-05-19 23:20:31.466166
